# K리그 패스 예측 - EDA

**목표:** 데이터 구조 이해 및 기본 통계 확인

In [2]:
# 모듈 경로 설정
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"✓ 프로젝트 루트: {project_root}")

✓ 프로젝트 루트: e:\Dacon\kleague-pass-prediction


In [3]:
# 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("✓ 라이브러리 로드 완료")

✓ 라이브러리 로드 완료


In [4]:
import torch

print(torch.version.cuda)
print(torch.cuda.is_available())

12.1
True


In [5]:
import pandas as pd

tabular = pd.read_parquet('../data/processed/v1/train_episodes.parquet')
sequence = pd.read_parquet('../data/processed/v1/train_sequences.parquet')

In [6]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 'game_id', 'episode_id', 'time_seconds'로 정렬한 후 action_id가 오름차순으로 잘 정렬되어 있는지 확인
df_sorted = sequence.sort_values(['game_id', 'episode_id', 'time_seconds', 'action_id'])
check = df_sorted.groupby(['game_id', 'episode_id', 'time_seconds'])['action_id'].apply(lambda x: (x.diff().dropna() >= 0).all())
print("모든 그룹에서 action_id가 오름차순인지 여부:", check.all())
print(check.value_counts())


모든 그룹에서 action_id가 오름차순인지 여부: True
action_id
True    329528
Name: count, dtype: int64


In [7]:
sequence

,game_id,period_id,episode_id,time_seconds,team_id,player_id,action_id,type_name,result_name,start_x,start_y,end_x,end_y,is_home,game_episode,dx,dy,distance,angle,dist_to_goal_start,dist_to_goal_end,goal_progress,zone_x,zone_y,dist_from_center,dist_to_touchline,action_order,action_order_norm,episode_length,time_diff,move_from_prev_x,move_from_prev_y,move_from_prev_dist,cumsum_distance,cumsum_goal_progress,speed,type_changed,is_pass,is_successful,distance_roll_3_mean,goal_progress_roll_3_mean,dx_roll_3_mean,dy_roll_3_mean,distance_roll_5_mean,goal_progress_roll_5_mean,dx_roll_5_mean,dy_roll_5_mean,home_score,away_score,score_diff,team_score_diff,is_winning,is_losing,is_draw,total_goals,type_name_encoded,result_name_encoded,zone_x_encoded,zone_y_encoded
0,126283,1,1,0.667000,2354,344559,0,Pass,Successful,52.418205,33.485443,31.322445,38.274754,True,126283_1,-21.095760,4.789308,21.632582,2.918350,52.584312,73.801460,-21.217148,midfield,center,0.514556,33.485443,0,0.000000,49,0.000,0.000000,0.000000,0.000000,21.632582,-21.217148,100.000000,1,1,1,21.632582,-21.217148,-21.095760,4.789308,21.632582,-21.217148,-21.095760,4.789308,1,0,1,1,1,0,0,1,16,5,2,0
1,126283,1,1,3.667000,2354,250036,2,Pass,Successful,32.013241,38.100807,37.371284,30.632980,True,126283_1,5.358045,-7.467828,9.191142,-0.948432,73.101875,67.712479,5.389393,defensive,center,4.100808,29.899193,1,0.020408,49,3.000,-20.404964,4.615364,20.920425,30.823723,-15.827756,3.053536,0,1,1,15.411861,-7.913878,-7.868857,-1.339260,15.411861,-7.913878,-7.868857,-1.339260,1,0,1,1,1,0,0,1,16,5,1,0
2,126283,1,1,4.968000,2354,500145,4,Carry,0,37.371284,30.632980,38.391571,24.613144,True,126283_1,1.020285,-6.019836,6.105687,-1.402905,67.712479,67.266602,0.445876,midfield,center,3.367020,30.632980,2,0.040816,49,1.301,5.358045,-7.467828,9.191142,36.929409,-15.381880,4.657274,1,0,0,12.309803,-5.127293,-4.905810,-2.899452,12.309803,-5.127293,-4.905810,-2.899452,1,0,1,1,1,0,0,1,2,-1,2,0
3,126283,1,1,8.200000,2354,500145,5,Pass,Successful,38.391571,24.613144,34.573349,5.545468,True,126283_1,-3.818220,-19.067677,19.446209,-1.768428,67.266602,75.957710,-8.691104,midfield,center,9.386856,24.613144,3,0.061224,49,3.232,1.020285,-6.019836,6.105687,56.375618,-24.072985,5.998214,1,1,1,11.581013,-0.951945,0.853370,-10.851780,14.093904,-6.018246,-4.633913,-6.941508,1,0,1,1,1,0,0,1,16,5,2,0
4,126283,1,1,11.633000,2354,142106,7,Pass,Successful,34.578705,6.058256,21.274469,18.437113,True,126283_1,-13.304235,12.378856,18.172472,2.392210,75.762123,85.159660,-9.397534,defensive,left,27.941744,6.058256,4,0.081633,49,3.433,-3.812865,-18.554888,18.942593,74.548088,-33.470516,5.278092,0,1,1,14.574789,-5.880921,-5.367390,-4.236218,14.909618,-6.694104,-6.367977,-3.077435,1,0,1,1,1,0,0,1,16,5,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356716,126480,2,85,2887.666992,4220,143588,2905,Pass,Successful,37.357109,20.782295,62.394466,4.904160,False,126480_85,25.037355,-15.878136,29.647671,-0.565172,68.922188,51.592632,17.329559,midfield,left,13.217704,20.782295,9,0.750000,12,0.000,0.000000,0.000000,0.000000,160.682510,125.670471,100.000000,1,1,1,23.382797,14.981547,19.469065,-12.903431,18.835224,13.038219,16.200975,-6.889447,1,1,0,0,0,0,1,2,16,5,2,1
356717,126480,2,85,2889.934082,4220,500530,2907,Carry,0,62.394466,4.904160,76.747017,17.890663,False,126480_85,14.352555,12.986504,19.355751,0.735473,51.592632,32.522938,19.069691,midfield,left,29.095840,4.904160,10,0.833333,12,2.267,25.037355,-15.878136,29.647671,180.038254,144.740158,8.500549,1,0,0,26.217031,17.909603,21.475756,-6.256589,19.453342,14.337403,16.058680,-5.519016,1,1,0,0,0,0,1,2,2,-1,2,1
356718,126480,2,85,2896.199951,4220,500530,2908,Pass,Successful,76.747017,17.890663,86.547722,25.628656,False,126480_85,9.800700,7.737992,12.487203,0.668327,32.522938,20.262428,12.2

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split



In [14]:
train = pd.read_csv('E:/Dacon/kleague-pass-prediction/data/raw/train.csv')
train[train['game_id'] == 126283].to_csv('raw_sample_data.csv', index = False)

In [49]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import optuna
from optuna.pruners import MedianPruner
from tqdm import tqdm
import copy

# 0. GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')  # RTX 3060이면 'cuda' 출력

# 1. 데이터 로드 및 전처리
sequence = pd.read_parquet('../data/processed/v1/train_sequences.parquet')

Using device: cuda


#### target 정규화 X

In [ ]:
# object 컬럼 LabelEncoding
obj_cols = sequence.select_dtypes(include='object').columns
label_encoders = {}
for col in obj_cols:
    le = LabelEncoder()
    sequence[col] = le.fit_transform(sequence[col])
    label_encoders[col] = le

# 수치 컬럼 MinMaxScaling (end_x, end_y 제외 → 타겟)
num_features = sequence.drop(columns=['end_x', 'end_y']).select_dtypes(exclude='object').columns
scalers = {}
for col in num_features:
    scaler = MinMaxScaler()
    sequence[col] = scaler.fit_transform(sequence[[col]])
    scalers[col] = scaler

input_size = len(num_features)

# 2. 에피소드별 시퀀스 준비 (전체 패딩 안 함 → 메모리 절약)
sequences = []  # list of tensor (variable length)
targets = []    # list of tensor [2]
for _, group in sequence.groupby('game_episode'):
    seq = group[num_features].values[:-1]  # 이전 이벤트들
    target = group[['end_x', 'end_y']].values[-1]  # 마지막 위치 (스케일링 안 된 원본)
    if len(seq) > 0:
        sequences.append(torch.tensor(seq, dtype=torch.float32))
        targets.append(torch.tensor(target, dtype=torch.float32))

# 3. Custom Dataset 정의 (한 에피소드 = 한 샘플)
class EpisodeDataset(Dataset):
    def __init__(self, seqs, targs):
        self.seqs = seqs
        self.targs = targs
    
    def __len__(self):
        return len(self.seqs)
    
    def __getitem__(self, idx):
        return self.seqs[idx], self.targs[idx]

# 4. Dynamic padding collate_fn (배치 내에서만 패딩 → OOM 해결 핵심)
def collate_fn(batch):
    seqs, targs = zip(*batch)
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=0.0)
    targs = torch.stack(targs)
    return seqs_padded, targs

# train/val split
train_seqs, val_seqs, train_targs, val_targs = train_test_split(
    sequences, targets, test_size=0.3, random_state=42)

train_dataset = EpisodeDataset(train_seqs, train_targs)
val_dataset = EpisodeDataset(val_seqs, val_targs)

In [53]:
# 5. 손실 함수
def euclidean_loss(preds, targets):
    diff = preds - targets
    dist = torch.sqrt(torch.sum(diff ** 2, dim=1) + 1e-8)
    return torch.mean(dist)

# 6. Early Stopping
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.best_model = None
    
    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_model = copy.deepcopy(model.state_dict())
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

# 7. 모델 정의
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_size, 2)
    
    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        return self.fc(hn[-1])

class TransformerModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_heads, dropout):
        super().__init__()
        self.embedding = nn.Linear(input_size, hidden_size)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=num_heads, dropout=dropout)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(hidden_size, 2)
    
    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        return self.fc(x[:, -1, :])

# 8. Optuna objective
def objective(trial):
    model_type = trial.suggest_categorical('model_type', ['LSTM', 'Transformer'])
    if model_type == 'Transformer' :
        num_heads = trial.suggest_int('num_heads', 2, 8)
    else :
        num_heads = 4

    min_hidden = ((64 + num_heads -1) // num_heads) * num_heads
    hidden_size = trial.suggest_int('hidden_size', min_hidden, 256, step = num_heads)
    num_layers = trial.suggest_int('num_layers', 1, 3)
    lr = trial.suggest_loguniform('lr', 1e-4, 1e-2)
    batch_size = trial.suggest_int('batch_size', 4, 32)  # OOM 방지 위해 낮춤
    dropout = trial.suggest_uniform('dropout', 0.1, 0.5)
    optimizer_type = trial.suggest_categorical('optimizer_type', ['Adam', 'RMSprop'])
    if model_type == 'Transformer':
        num_heads = trial.suggest_int('num_heads', 2, 8)
    
    # 모델 생성 및 GPU 이동
    if model_type == 'LSTM':
        model = LSTMModel(input_size, hidden_size, num_layers, dropout)
    else:
        model = TransformerModel(input_size, hidden_size, num_layers, num_heads, dropout)
    model.to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=lr) if optimizer_type == 'Adam' else optim.RMSprop(model.parameters(), lr=lr)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              pin_memory=True, collate_fn=collate_fn)
    
    early_stopping = EarlyStopping(patience=5)
    
    epochs = 50
    for epoch in tqdm(range(epochs), desc=f'Trial {trial.number}', leave=False):
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = euclidean_loss(outputs, batch_y)
            loss.backward()
            optimizer.step()
        
        # val_loss (배치로 계산 → 안정적)
        model.eval()
        val_loss = 0
        count = 0
        with torch.no_grad():
            for batch_x, batch_y in DataLoader(val_dataset, batch_size=32, collate_fn=collate_fn, pin_memory=True):
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                val_loss += euclidean_loss(outputs, batch_y).item() * batch_x.size(0)
                count += batch_x.size(0)
        val_loss /= count
        
        trial.report(val_loss, epoch)
        if early_stopping(val_loss, model):
            break
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return val_loss

In [54]:
# 9. Optuna 실행
study = optuna.create_study(direction='minimize', pruner=MedianPruner())
n_trials = 50
for _ in tqdm(range(n_trials), desc='Optuna Trials'):
    study.optimize(objective, n_trials=1)

print("Best HyperParams:", study.best_params)
print("Best Val Distance:", study.best_value)

[I 2025-12-24 13:12:43,666] A new study created in memory with name: no-name-0b1fc8e0-d4fb-4a9c-ab05-3fe30a4517ae
Optuna Trials:   0%|          | 0/50 [00:00<?, ?it/s]C:\Users\leo99\AppData\Local\Temp\ipykernel_15548\2827595755.py:62: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform('lr', 1e-4, 1e-2)
C:\Users\leo99\AppData\Local\Temp\ipykernel_15548\2827595755.py:64: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  dropout = trial.suggest_uniform('dropout', 0.1, 0.5)
[I 2025-12-24 13:15:39,376] Trial 0 finished with value: 31.898461983615416 and parameters: {'model_type': 'LSTM', 'hidden_size': 148, 'num_layers': 3, 'lr': 0.00011368624088734091, 'batch_size': 12, 

Best HyperParams: {'model_type': 'LSTM', 'hidden_size': 176, 'num_layers': 3, 'lr': 0.0006028123377202644, 'batch_size': 14, 'dropout': 0.16282086082709554, 'optimizer_type': 'RMSprop'}
Best Val Distance: 16.405115627784095


#### target 정규화 O

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import optuna
from optuna.pruners import MedianPruner
from tqdm import tqdm
import copy

# 0. GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')  # RTX 3060이면 'cuda' 출력

# 1. 데이터 로드 및 전처리
sequence = pd.read_parquet('../data/processed/v1/train_sequences.parquet')
# object 컬럼 LabelEncoding
obj_cols = sequence.select_dtypes(include='object').columns
label_encoders = {}
for col in obj_cols:
    le = LabelEncoder()
    sequence[col] = le.fit_transform(sequence[col])
    label_encoders[col] = le

# 수치 컬럼 MinMaxScaling (end_x, end_y 제외 → 타겟)
num_features = sequence.select_dtypes(exclude='object').columns
scalers = {}
for col in num_features:
    scaler = MinMaxScaler()
    sequence[col] = scaler.fit_transform(sequence[[col]])
    scalers[col] = scaler

input_size = len(num_features)

# 2. 에피소드별 시퀀스 준비 (전체 패딩 안 함 → 메모리 절약)
sequences = []  # list of tensor (variable length)
targets = []    # list of tensor [2]
for _, group in sequence.groupby('game_episode'):
    seq = group[num_features].values[:-1]  # 이전 이벤트들
    target = group[['end_x', 'end_y']].values[-1]  # 마지막 위치 (스케일링 안 된 원본)
    if len(seq) > 0:
        sequences.append(torch.tensor(seq, dtype=torch.float32))
        targets.append(torch.tensor(target, dtype=torch.float32))

# 3. Custom Dataset 정의 (한 에피소드 = 한 샘플)
class EpisodeDataset(Dataset):
    def __init__(self, seqs, targs):
        self.seqs = seqs
        self.targs = targs
    
    def __len__(self):
        return len(self.seqs)
    
    def __getitem__(self, idx):
        return self.seqs[idx], self.targs[idx]

# 4. Dynamic padding collate_fn (배치 내에서만 패딩 → OOM 해결 핵심)
def collate_fn(batch):
    seqs, targs = zip(*batch)
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=0.0)
    targs = torch.stack(targs)
    return seqs_padded, targs

# train/val split
train_seqs, val_seqs, train_targs, val_targs = train_test_split(
    sequences, targets, test_size=0.3, random_state=42)

train_dataset = EpisodeDataset(train_seqs, train_targs)
val_dataset = EpisodeDataset(val_seqs, val_targs)
# 5. 손실 함수
def euclidean_loss(preds, targets):
    diff = preds - targets
    dist = torch.sqrt(torch.sum(diff ** 2, dim=1) + 1e-8)
    return torch.mean(dist)

# 6. Early Stopping
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.best_model = None
    
    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_model = copy.deepcopy(model.state_dict())
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

# 7. 모델 정의
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_size, 2)
    
    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        return self.fc(hn[-1])

class TransformerModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_heads, dropout):
        super().__init__()
        self.embedding = nn.Linear(input_size, hidden_size)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=num_heads, dropout=dropout)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(hidden_size, 2)
    
    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        return self.fc(x[:, -1, :])

# 8. Optuna objective
def objective(trial):
    model_type = trial.suggest_categorical('model_type', ['LSTM', 'Transformer'])
    if model_type == 'Transformer' :
        num_heads = trial.suggest_int('num_heads', 2, 8)
    else :
        num_heads = 4

    min_hidden = ((64 + num_heads -1) // num_heads) * num_heads
    hidden_size = trial.suggest_int('hidden_size', min_hidden, 256, step = num_heads)
    num_layers = trial.suggest_int('num_layers', 1, 3)
    lr = trial.suggest_loguniform('lr', 1e-4, 1e-2)
    batch_size = trial.suggest_int('batch_size', 4, 32)  # OOM 방지 위해 낮춤
    dropout = trial.suggest_uniform('dropout', 0.1, 0.5)
    optimizer_type = trial.suggest_categorical('optimizer_type', ['Adam', 'RMSprop'])
    if model_type == 'Transformer':
        num_heads = trial.suggest_int('num_heads', 2, 8)
    
    # 모델 생성 및 GPU 이동
    if model_type == 'LSTM':
        model = LSTMModel(input_size, hidden_size, num_layers, dropout)
    else:
        model = TransformerModel(input_size, hidden_size, num_layers, num_heads, dropout)
    model.to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=lr) if optimizer_type == 'Adam' else optim.RMSprop(model.parameters(), lr=lr)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              pin_memory=True, collate_fn=collate_fn)
    
    early_stopping = EarlyStopping(patience=5)
    
    epochs = 50
    for epoch in tqdm(range(epochs), desc=f'Trial {trial.number}', leave=False):
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = euclidean_loss(outputs, batch_y)
            loss.backward()
            optimizer.step()
        
        # val_loss (배치로 계산 → 안정적)
        model.eval()
        val_loss = 0
        count = 0
        with torch.no_grad():
            for batch_x, batch_y in DataLoader(val_dataset, batch_size=32, collate_fn=collate_fn, pin_memory=True):
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                val_loss += euclidean_loss(outputs, batch_y).item() * batch_x.size(0)
                count += batch_x.size(0)
        val_loss /= count
        
        trial.report(val_loss, epoch)
        if early_stopping(val_loss, model):
            break
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return val_loss
# 9. Optuna 실행
study = optuna.create_study(direction='minimize', pruner=MedianPruner())
n_trials = 50
for _ in tqdm(range(n_trials), desc='Optuna Trials'):
    study.optimize(objective, n_trials=1)

print("Best HyperParams:", study.best_params)
print("Best Val Distance:", study.best_value)

In [56]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import optuna
from optuna.pruners import MedianPruner
from tqdm import tqdm
import copy

# 0. GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 1. 데이터 로드
sequence = pd.read_parquet('../data/processed/v1/train_sequences.parquet')

# 2. object 컬럼 LabelEncoding
obj_cols = sequence.select_dtypes(include='object').columns
label_encoders = {}
for col in obj_cols:
    le = LabelEncoder()
    sequence[col] = le.fit_transform(sequence[col])
    label_encoders[col] = le

# 3. 타겟 정규화 scaler
target_scaler = MinMaxScaler()
all_targets = sequence[['end_x', 'end_y']].values
target_scaler.fit(all_targets)

# 4. 입력 피처 스케일링 (타겟 제외)
num_features = sequence.drop(columns=['end_x', 'end_y'], errors='ignore').select_dtypes(exclude='object').columns
scalers = {}
for col in num_features:
    scaler = MinMaxScaler()
    sequence[col] = scaler.fit_transform(sequence[[col]])
    scalers[col] = scaler

input_size = len(num_features)

# 5. 에피소드별 시퀀스 준비
sequences = []
targets = []
for _, group in sequence.groupby('game_episode'):
    seq = group[num_features].values[:-1]
    target_original = group[['end_x', 'end_y']].values[-1]
    target_norm = target_scaler.transform(target_original.reshape(1, -1)).flatten()
    if len(seq) > 0:
        sequences.append(torch.tensor(seq, dtype=torch.float32))
        targets.append(torch.tensor(target_norm, dtype=torch.float32))

# 6. Custom Dataset 및 collate_fn
class EpisodeDataset(Dataset):
    def __init__(self, seqs, targs):
        self.seqs = seqs
        self.targs = targs
    
    def __len__(self):
        return len(self.seqs)
    
    def __getitem__(self, idx):
        return self.seqs[idx], self.targs[idx]

def collate_fn(batch):
    seqs, targs = zip(*batch)
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=0.0)
    targs = torch.stack(targs)
    return seqs_padded, targs

# train/val split
train_seqs, val_seqs, train_targs, val_targs = train_test_split(
    sequences, targets, test_size=0.3, random_state=42)

train_dataset = EpisodeDataset(train_seqs, train_targs)
val_dataset = EpisodeDataset(val_seqs, val_targs)

val_loader = DataLoader(val_dataset, batch_size=32, collate_fn=collate_fn, pin_memory=True)

# 7. 손실 함수
def euclidean_loss(preds, targets):
    diff = preds - targets
    dist = torch.sqrt(torch.sum(diff ** 2, dim=1) + 1e-8)
    return torch.mean(dist)

# 8. Early Stopping
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.best_model = None
    
    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_model = copy.deepcopy(model.state_dict())
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

# 9. 모델 정의
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_size, 2)
    
    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        return self.fc(hn[-1])

class TransformerModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_heads, dropout):
        super().__init__()
        self.embedding = nn.Linear(input_size, hidden_size)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=num_heads, dropout=dropout)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(hidden_size, 2)
    
    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        return self.fc(x[:, -1, :])

# 10. Optuna objective
def objective(trial):
    model_type = trial.suggest_categorical('model_type', ['LSTM', 'Transformer'])
    
    if model_type == 'Transformer':
        num_heads = trial.suggest_int('num_heads', 2, 8)
    else:
        num_heads = 4
    
    min_hidden = ((64 + num_heads - 1) // num_heads) * num_heads
    hidden_size = trial.suggest_int('hidden_size', min_hidden, 256, step=num_heads)
    
    num_layers = trial.suggest_int('num_layers', 1, 3)
    lr = trial.suggest_loguniform('lr', 1e-4, 1e-2)
    batch_size = trial.suggest_int('batch_size', 4, 32)
    dropout = trial.suggest_uniform('dropout', 0.1, 0.5)
    optimizer_type = trial.suggest_categorical('optimizer_type', ['Adam', 'RMSprop'])
    
    if model_type == 'LSTM':
        model = LSTMModel(input_size, hidden_size, num_layers, dropout)
    else:
        model = TransformerModel(input_size, hidden_size, num_layers, num_heads, dropout)
    model.to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=lr) if optimizer_type == 'Adam' else optim.RMSprop(model.parameters(), lr=lr)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              pin_memory=True, collate_fn=collate_fn)
    
    early_stopping = EarlyStopping(patience=5)
    
    epochs = 50
    for epoch in tqdm(range(epochs), desc=f'Trial {trial.number}', leave=False):
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = euclidean_loss(outputs, batch_y)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss_norm = 0
        count = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                val_loss_norm += euclidean_loss(outputs, batch_y).item() * batch_x.size(0)
                count += batch_x.size(0)
        val_loss_norm /= count
        
        trial.report(val_loss_norm, epoch)
        if early_stopping(val_loss_norm, model):
            break
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return val_loss_norm

# 11. Optuna 실행
study = optuna.create_study(direction='minimize', pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5))
n_trials = 50
for _ in tqdm(range(n_trials), desc='Optuna Trials'):
    study.optimize(objective, n_trials=1)

print("Best HyperParams:", study.best_params)
print("Best Val Distance (normalized space):", study.best_value)

# 12. 최종 모델 학습 (best_params 사용)
best_params = study.best_params
model_type = best_params['model_type']

if model_type == 'Transformer':
    num_heads = best_params['num_heads']
else:
    num_heads = 4

if model_type == 'LSTM':
    final_model = LSTMModel(input_size, best_params['hidden_size'], best_params['num_layers'], best_params['dropout'])
else:
    final_model = TransformerModel(input_size, best_params['hidden_size'], best_params['num_layers'], num_heads, best_params['dropout'])

final_model.to(device)

optimizer = optim.Adam(final_model.parameters(), lr=best_params['lr']) if best_params['optimizer_type'] == 'Adam' else optim.RMSprop(final_model.parameters(), lr=best_params['lr'])

train_loader = DataLoader(train_dataset, batch_size=best_params['batch_size'], shuffle=True,
                          pin_memory=True, collate_fn=collate_fn)

early_stopping = EarlyStopping(patience=10)

epochs = 200
for epoch in tqdm(range(epochs), desc='Final Model Training'):
    final_model.train()
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = final_model(batch_x)
        loss = euclidean_loss(outputs, batch_y)
        loss.backward()
        optimizer.step()
    
    final_model.eval()
    val_loss_norm = 0
    count = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = final_model(batch_x)
            val_loss_norm += euclidean_loss(outputs, batch_y).item() * batch_x.size(0)
            count += batch_x.size(0)
    val_loss_norm /= count
    
    if early_stopping(val_loss_norm, final_model):
        print(f'Final training stopped early at epoch {epoch+1}')
        break
    
    if (epoch+1) % 10 == 0:
        print(f'Final Epoch {epoch+1}, Val Loss (norm): {val_loss_norm:.6f}')

# best_model 로드
if early_stopping.best_model is not None:
    final_model.load_state_dict(early_stopping.best_model)

# 최종 원본 스케일 거리 계산
final_model.eval()
total_dist = 0
count = 0
with torch.no_grad():
    for batch_x, batch_y in val_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        pred_norm = final_model(batch_x)
        pred_orig = target_scaler.inverse_transform(pred_norm.cpu().numpy())
        true_orig = target_scaler.inverse_transform(batch_y.cpu().numpy())
        batch_dist = np.sqrt(np.sum((pred_orig - true_orig)**2, axis=1))
        total_dist += batch_dist.sum()
        count += batch_x.size(0)

final_original_distance = total_dist / count
print(f'Final Validation Euclidean Distance (original scale): {final_original_distance:.4f}')

Using device: cuda


[I 2025-12-24 14:54:48,494] A new study created in memory with name: no-name-ea70f28d-f981-4a18-82a9-ff6eeca129a1
Optuna Trials:   0%|          | 0/50 [00:00<?, ?it/s]e:\Dacon\kleague-pass-prediction\venv\Lib\site-packages\optuna\distributions.py:684: UserWarning: The distribution is specified by [66, 256] and step=3, but the range is not divisible by `step`. It will be replaced with [66, 255].
  warnings.warn(
C:\Users\leo99\AppData\Local\Temp\ipykernel_15548\3732912450.py:146: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform('lr', 1e-4, 1e-2)
C:\Users\leo99\AppData\Local\Temp\ipykernel_15548\3732912450.py:148: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  dro

Best HyperParams: {'model_type': 'LSTM', 'hidden_size': 100, 'num_layers': 2, 'lr': 0.0006849110466722001, 'batch_size': 4, 'dropout': 0.3191292777691463, 'optimizer_type': 'RMSprop'}
Best Val Distance (normalized space): 0.18858498419546674


Final Model Training:   5%|▌         | 10/200 [01:44<33:03, 10.44s/it]

Final Epoch 10, Val Loss (norm): 0.195142


Final Model Training:  10%|█         | 20/200 [03:28<30:41, 10.23s/it]

Final Epoch 20, Val Loss (norm): 0.189250


Final Model Training:  14%|█▍        | 29/200 [05:11<30:38, 10.75s/it]

Final training stopped early at epoch 30


Final Validation Euclidean Distance (original scale): 15.5014


#### 4. 학습 강화

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import optuna
from optuna.pruners import MedianPruner
from tqdm import tqdm
import copy

# 0. GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 1. 데이터 로드
sequence = pd.read_parquet('../data/processed/v1/train_sequences.parquet')

# 2. object 컬럼 LabelEncoding
obj_cols = sequence.select_dtypes(include='object').columns
label_encoders = {}
for col in obj_cols:
    le = LabelEncoder()
    sequence[col] = le.fit_transform(sequence[col])
    label_encoders[col] = le

# 3. 타겟 정규화 scaler
target_scaler = MinMaxScaler()
all_targets = sequence[['end_x', 'end_y']].values
target_scaler.fit(all_targets)

# 4. 입력 피처 스케일링 (타겟 제외)
num_features = sequence.drop(columns=['end_x', 'end_y'], errors='ignore').select_dtypes(exclude='object').columns
scalers = {}
for col in num_features:
    scaler = MinMaxScaler()
    sequence[col] = scaler.fit_transform(sequence[[col]])
    scalers[col] = scaler

input_size = len(num_features)

# 5. 에피소드별 시퀀스 준비
sequences = []
targets = []
for _, group in sequence.groupby('game_episode'):
    seq = group[num_features].values[:-1]
    target_original = group[['end_x', 'end_y']].values[-1]
    target_norm = target_scaler.transform(target_original.reshape(1, -1)).flatten()
    if len(seq) > 0:
        sequences.append(torch.tensor(seq, dtype=torch.float32))
        targets.append(torch.tensor(target_norm, dtype=torch.float32))

# 6. Custom Dataset 및 collate_fn
class EpisodeDataset(Dataset):
    def __init__(self, seqs, targs):
        self.seqs = seqs
        self.targs = targs
    
    def __len__(self):
        return len(self.seqs)
    
    def __getitem__(self, idx):
        return self.seqs[idx], self.targs[idx]

def collate_fn(batch):
    seqs, targs = zip(*batch)
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=0.0)
    targs = torch.stack(targs)
    return seqs_padded, targs

# train/val split
train_seqs, val_seqs, train_targs, val_targs = train_test_split(
    sequences, targets, test_size=0.3, random_state=42)

train_dataset = EpisodeDataset(train_seqs, train_targs)
val_dataset = EpisodeDataset(val_seqs, val_targs)

val_loader = DataLoader(val_dataset, batch_size=32, collate_fn=collate_fn, pin_memory=True)

# 7. 손실 함수
def euclidean_loss(preds, targets):
    diff = preds - targets
    dist = torch.sqrt(torch.sum(diff ** 2, dim=1) + 1e-8)
    return torch.mean(dist)

# 8. Early Stopping
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.best_model = None
    
    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_model = copy.deepcopy(model.state_dict())
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

# 9. 모델 정의
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_size, 2)
    
    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        return self.fc(hn[-1])

class TransformerModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_heads, dropout):
        super().__init__()
        self.embedding = nn.Linear(input_size, hidden_size)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=num_heads, dropout=dropout)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(hidden_size, 2)
    
    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        return self.fc(x[:, -1, :])

# 10. Optuna objective
def objective(trial):
    model_type = trial.suggest_categorical('model_type', ['LSTM', 'Transformer'])
    
    if model_type == 'Transformer':
        num_heads = trial.suggest_int('num_heads', 2, 8)
    else:
        num_heads = 4
    
    min_hidden = ((64 + num_heads - 1) // num_heads) * num_heads
    hidden_size = trial.suggest_int('hidden_size', min_hidden, 256, step=num_heads)
    
    num_layers = trial.suggest_int('num_layers', 1, 3)
    lr = trial.suggest_loguniform('lr', 1e-4, 1e-2)
    batch_size = trial.suggest_int('batch_size', 4, 32)
    dropout = trial.suggest_uniform('dropout', 0.1, 0.5)
    optimizer_type = trial.suggest_categorical('optimizer_type', ['Adam', 'RMSprop'])
    
    if model_type == 'LSTM':
        model = LSTMModel(input_size, hidden_size, num_layers, dropout)
    else:
        model = TransformerModel(input_size, hidden_size, num_layers, num_heads, dropout)
    model.to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=lr) if optimizer_type == 'Adam' else optim.RMSprop(model.parameters(), lr=lr)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              pin_memory=True, collate_fn=collate_fn)
    
    early_stopping = EarlyStopping(patience=5)
    
    epochs = 50
    for epoch in tqdm(range(epochs), desc=f'Trial {trial.number}', leave=False):
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = euclidean_loss(outputs, batch_y)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss_norm = 0
        count = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                val_loss_norm += euclidean_loss(outputs, batch_y).item() * batch_x.size(0)
                count += batch_x.size(0)
        val_loss_norm /= count
        
        trial.report(val_loss_norm, epoch)
        if early_stopping(val_loss_norm, model):
            break
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return val_loss_norm

# 11. Optuna 실행
study = optuna.create_study(direction='minimize', pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5))
n_trials = 50
for _ in tqdm(range(n_trials), desc='Optuna Trials'):
    study.optimize(objective, n_trials=1)

print("Best HyperParams:", study.best_params)
print("Best Val Distance (normalized space):", study.best_value)

In [57]:
# 12. 최종 모델 학습 (강화 버전)
best_params = study.best_params
model_type = best_params['model_type']

if model_type == 'Transformer':
    num_heads = best_params['num_heads']
else:
    num_heads = 4  # dummy

if model_type == 'LSTM':
    final_model = LSTMModel(input_size, best_params['hidden_size'], best_params['num_layers'], best_params['dropout'])
else:
    final_model = TransformerModel(input_size, best_params['hidden_size'], best_params['num_layers'], num_heads, best_params['dropout'])

final_model.to(device)

optimizer = optim.Adam(final_model.parameters(), lr=best_params['lr']) if best_params['optimizer_type'] == 'Adam' else optim.RMSprop(final_model.parameters(), lr=best_params['lr'])

# scheduler 추가 (ReduceLROnPlateau)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, verbose=True)

train_loader = DataLoader(train_dataset, batch_size=best_params['batch_size'], shuffle=True,
                          pin_memory=True, collate_fn=collate_fn)

early_stopping = EarlyStopping(patience=30)  # patience 늘림

epochs = 500  # 크게 늘림 (early stopping이 알아서 멈춤)
for epoch in tqdm(range(epochs), desc='Final Model Training'):
    final_model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = final_model(batch_x)
        loss = euclidean_loss(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    final_model.eval()
    val_loss_norm = 0
    count = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = final_model(batch_x)
            val_loss_norm += euclidean_loss(outputs, batch_y).item() * batch_x.size(0)
            count += batch_x.size(0)
    val_loss_norm /= count
    
    # scheduler step (val_loss_norm으로 lr 조정)
    scheduler.step(val_loss_norm)
    
    if early_stopping(val_loss_norm, final_model):
        print(f'Final training stopped early at epoch {epoch+1}')
        break
    
    if (epoch+1) % 10 == 0:
        # 원본 스케일 거리 로그
        with torch.no_grad():
            sample_x, sample_y = next(iter(val_loader))
            sample_x, sample_y = sample_x.to(device), sample_y.to(device)
            sample_pred_norm = final_model(sample_x)
            sample_pred_orig = target_scaler.inverse_transform(sample_pred_norm.cpu().numpy())
            sample_true_orig = target_scaler.inverse_transform(sample_y.cpu().numpy())
            sample_dist = np.sqrt(np.sum((sample_pred_orig - sample_true_orig)**2, axis=1)).mean()
        print(f'Final Epoch {epoch+1}, Val Loss (norm): {val_loss_norm:.6f}, Approx Original Distance: {sample_dist:.4f}')

# best_model 로드
if early_stopping.best_model is not None:
    final_model.load_state_dict(early_stopping.best_model)

# 최종 원본 거리 계산
final_model.eval()
total_dist = 0
count = 0
with torch.no_grad():
    for batch_x, batch_y in val_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        pred_norm = final_model(batch_x)
        pred_orig = target_scaler.inverse_transform(pred_norm.cpu().numpy())
        true_orig = target_scaler.inverse_transform(batch_y.cpu().numpy())
        batch_dist = np.sqrt(np.sum((pred_orig - true_orig)**2, axis=1))
        total_dist += batch_dist.sum()
        count += batch_x.size(0)

final_original_distance = total_dist / count
print(f'Final Validation Euclidean Distance (original scale): {final_original_distance:.4f}')

e:\Dacon\kleague-pass-prediction\venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Final Model Training:   2%|▏         | 10/500 [01:37<1:22:58, 10.16s/it]

Final Epoch 10, Val Loss (norm): 0.206684, Approx Original Distance: 18.0544


Final Model Training:   4%|▍         | 20/500 [03:23<1:21:01, 10.13s/it]

Final Epoch 20, Val Loss (norm): 0.197602, Approx Original Distance: 17.2066


Final Model Training:   6%|▌         | 30/500 [05:12<1:23:38, 10.68s/it]

...

Final Model Training:  12%|█▏        | 60/500 [10:41<1:25:11, 11.62s/it]

Final Epoch 60, Val Loss (norm): 0.204256, Approx Original Distance: 18.7735


Final Model Training:  12%|█▏        | 60/500 [10:52<1:19:43, 10.87s/it]

Final training stopped early at epoch 61


Final Validation Euclidean Distance (original scale): 15.7991
